In [6]:
!git clone https://github.com/sooo66/semeval2026-task12-dataset.git

Cloning into 'semeval2026-task12-dataset'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 22 (delta 6), reused 16 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 4.25 MiB | 10.11 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [7]:
example_task = {
    "topic_id": 3,
    "uuid": "0951a9fb-1af0-45ff-aee1-64791e053a35",
    "target_event": "Capitol Police Chief Steven Sund, House Sergeant at Arms Paul Irving, and Senate Sergeant at Arms Michael Stenger resigned.",
    "option_A": "The congressional certification of Joe Biden’s victory was halted.",
    "option_B": "Protesters clashed with police and breached security barriers at the Capitol.",
    "option_C": "The Capitol was secured, and lawmakers reconvened to continue certification.",
    "option_D": "The D.C. National Guard was activated, and a citywide curfew was imposed.",
    "golden_answer": "B,C"
}

In [8]:
target_event = example_task["target_event"]
options = {
    "A": example_task["option_A"],
    "B": example_task["option_B"],
    "C": example_task["option_C"],
    "D": example_task["option_D"],
}

# prompt template
prompt = f"Given this event <event>{target_event}</event> Out of the following options what could be the cause of the event? <A>{options["A"]}</A> <B>{options["B"]}</B> <C>{options["C"]}</C> <D>{options["D"]}</D> Return options A,B,C or D (multiple options can be true) in this format in <answer> ... </answer> (multiple options seperated by commas ',')"
messages = [
    {"role": "system", "content": "You are helpful agent with reasonable thinking."},
    {"role": "user", "content": prompt},
]

In [9]:
import json

with open('/content/semeval2026-task12-dataset/sample_data/questions.jsonl', 'r') as json_file:
    json_list = list(json_file)

tasks = []
for json_str in json_list:
    result = json.loads(json_str)
    tasks.append(result)

In [10]:
from typing import Dict, List, Tuple

# Load and Index the Documents
def load_docs_map(file_path: str) -> Dict[int, List[Dict]]:
    """Loads docs.json and creates a dictionary mapping topic_id -> list of docs."""
    with open(file_path, 'r') as f:
        docs_data = json.load(f)
    docs_map = {int(item['topic_id']): item['docs'] for item in docs_data}
    return docs_map

# Load the docs
docs_path = '/content/semeval2026-task12-dataset/sample_data/docs.json'
try:
    if 'docs_map' not in globals():
        docs_map = load_docs_map(docs_path)
except NameError:
    docs_map = load_docs_map(docs_path)
print(f"Loaded documents for {len(docs_map)} topics.")

Loaded documents for 10 topics.


In [11]:
from typing import Set

def extract_answer(text: str) -> Set[str]:
    if not isinstance(text, str): return set()
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
    content = match.group(1) if match else text
    return set(re.findall(r'\b([A-D])\b', content.upper()))

In [12]:
def get_base_prompt(task: dict) -> str:
    target_event = task["target_event"]
    options = {
        "A": task["option_A"],
        "B": task["option_B"],
        "C": task["option_C"],
        "D": task["option_D"],
    }
    prompt = f"Given this event <event>{target_event}</event> Out of the following options what could be the cause of the event? <A>{options["A"]}</A> <B>{options["B"]}</B> <C>{options["C"]}</C> <D>{options["D"]}</D> Return options A,B,C or D (multiple options can be true) in this format in <answer> ... </answer> (multiple options seperated by commas ',')"
    return prompt

def get_base_prompt_message(task: dict) -> list[dict]:
    prompt = get_base_prompt(task)
    messages = [
        {"role": "system", "content": "You are helpful agent with reasonable thinking."},
        {"role": "user", "content": prompt},
    ]
    return messages

#RAG

In [16]:
from sentence_transformers import SentenceTransformer
retriever_model = SentenceTransformer('all-MiniLM-L6-v2')

In [17]:
import re
import time
from typing import List
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

# --- Configuration ---
MODEL_NAME = "gemini-2.5-flash-lite"
RETRIEVER_TOP_K = 10
MAX_CHAR_PER_DOC = 10000

# --- RAG Utils ---

def format_context_optimized(docs: List[Dict], target_event: str, top_k: int = RETRIEVER_TOP_K) -> str:
    """
    Sorts documents by similarity and formats them for the prompt.
    Retrieves more documents (top_k=10) to reduce the chance of missing the golden doc.
    """
    if not docs:
        return "No relevant documents found."

    selected_docs = []

    # Semantic Sorting
    if 'retriever_model' in globals() and retriever_model:
        try:
            doc_texts = [doc.get("text", doc.get("content", "")) for doc in docs]
            query_embedding = retriever_model.encode([target_event])
            doc_embeddings = retriever_model.encode(doc_texts)
            similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

            # Sort by similarity desc
            scored_docs = sorted(zip(similarities, docs), key=lambda x: x[0], reverse=True)
            selected_docs = [doc for score, doc in scored_docs[:top_k]]
        except Exception as e:
            print(f"Embedding failed: {e}. Fallback to list slicing.")
            selected_docs = docs[:top_k]
    else:
        selected_docs = docs[:top_k]

    # Formatting
    context_parts = []
    for i, doc in enumerate(selected_docs):
        content = doc.get("text", doc.get("content", str(doc)))
        # Relaxed truncation
        if len(content) > MAX_CHAR_PER_DOC:
            content = content[:MAX_CHAR_PER_DOC] + "...[truncated]"
        context_parts.append(f"Document {i+1}:\n{content}\n")

    return "\n".join(context_parts)

In [18]:
def get_rag_prompt(task: dict, docs_map: dict) -> str:
    """Minimal RAG Prompt with Context and strict answer format."""
    target_event = task["target_event"]
    topic_id = int(task["topic_id"])
    related_docs = docs_map.get(topic_id, [])

    context_str = format_context_optimized(related_docs, target_event)
    options = {
        "A": task["option_A"],
        "B": task["option_B"],
        "C": task["option_C"],
        "D": task["option_D"],
    }

    prompt = (
        f"CONTEXT (May contain distractors): {context_str}\n\n"
        f"Use the context above to analyze the following question.\n"
        f"Given this event <event>{target_event}</event> Out of the following options what could be the cause of the event? "
        f"<A>{options['A']}</A> <B>{options['B']}</B> <C>{options['C']}</C> <D>{options['D']}</D> "
        f"FIRST: Provide your step-by-step thinking, using the Context, in <reasoning>...</reasoning>. "
        f"SECOND: Return options A,B,C or D (multiple options can be true) in this format in <answer> ... </answer> (multiple options seperated by commas ','). "
        f"CRITICAL RULE: Include ALL correct options, even if they are textually identical."
    )
    return prompt

def get_task_prompt_message(task: dict, is_rag: bool = False, docs_map: dict = None) -> list[dict]:
    """Helper function for message format."""

    if is_rag:
        prompt = get_rag_prompt(task, docs_map)
    else:
        prompt = get_base_prompt(task)

    messages = [
        {"role": "system", "content": "You are helpful agent with reasonable thinking."},
        {"role": "user", "content": prompt},
    ]
    return messages

#Gemini

In [19]:
import os
from google.colab import userdata
from google import genai
from google.genai import types

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
client = genai.Client()

In [20]:
def get_base_response(task: dict) -> str:
    prompt = get_base_prompt(task)
    base_response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        config=types.GenerateContentConfig(
            system_instruction="You are helpful agent with reasonable thinking."),
        contents=prompt
    )
    return base_response.text

In [21]:
import time

for task in tasks:
    if "base_response" in task:
        continue

    base_response = get_base_response(task)
    task["base_response"] = base_response

    time.sleep(5)

with open('/content/gemini-flash-lite-base-output.jsonl', 'w') as f:
    for task in tasks:
        f.write(json.dumps(task) + '\n')

In [28]:
def get_rag_response(task: dict, docs_map: dict) -> str:
    prompt = get_rag_prompt(task, docs_map)
    rag_response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        config=types.GenerateContentConfig(
            system_instruction="You are helpful agent with reasonable thinking. Use the documents critically."),
        contents=prompt
    )
    return rag_response.text

In [30]:
for task in tasks:
    if "rag_response" in task:
        continue

    rag_response = get_rag_response(task, docs_map)
    task["rag_response"] = rag_response

    time.sleep(5)

with open('/content/gemini-flash-lite-rag-output.jsonl', 'w') as f:
    for task in tasks:
        f.write(json.dumps(task) + '\n')

#OpenRouter

In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=userdata.get("OPENROUTER_API_KEY"),
)

completion = client.chat.completions.create(
  model="qwen/qwen3-14b:free",
  messages=messages
)

print(completion.choices[0].message.content)

In [ ]:
def get_response(task) -> str:
    messages = get_task_prompt_message(task)
    completion = client.chat.completions.create(
        model="qwen/qwen3-14b:free",
        messages=messages
    )
    return completion.choices[0].message.content

In [ ]:
for task in tasks:
    if "response" in task:
        continue

    messages = get_task_prompt_message(task)
    response = get_response(messages)
    task["response"] = response

In [ ]:
with open('/content/qwen3_14b_output.jsonl', 'w') as f:
    for task in tasks:
        f.write(json.dumps(task) + '\n')

#Local

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [13]:
def run_one_task(model, tokenizer, task):

    messages = get_base_prompt_message(task)

    # prepare model input
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2048
    )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    answer = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    cleaned_answer = extract_choice(answer)

    return thinking, cleaned_answer


In [17]:
import re

def extract_choice(text):
    m = re.findall(r"[A-D]", text)
    if m:
        return ",".join(m)

    return ""

In [18]:
thinking, answer = run_one_task(model, tokenizer, tasks[0])
print(thinking)
print(answer)

<think>
Okay, let's tackle this question. The event is "Videos of the assassination circulated on social media." The question asks what could be the cause of this event, with options A to D.

First, I need to understand what each option implies. Let's go through them one by one.

Option A says the shooter used a handmade gun. If that's true, then the video would be a result of the shooter's action, which could be the cause. But does the presence of the video mean that the cause is the shooter's gun? Maybe, but I'm not sure if that's the direct cause.

Option B states that security arrested the suspected gunman, Tetsuya Yamagami. If the videos circulated, it might indicate that the arrest happened, but does that directly cause the video? Or maybe the arrest was part of the investigation, leading to the videos being shared? Not sure yet.

Option C mentions Shinzo Abe becoming the deputy chief cabinet secretary in the early 2000s. This seems unrelated to the assassination. Unless there wa

In [19]:
from tqdm import tqdm

def run_model_on_dataset(model, tokenizer, tasks, output_path):
    results = []

    for task in tqdm(tasks):
        thinking, answer = run_one_task(model, tokenizer, task)

        result = {
            "topic_id": task["topic_id"],
            "target_event": task.get("target_event", ""),
            "golden_answer": task.get("golden_answer", ""),
            "model_answer": answer,
            "thinking": thinking,
        }

        results.append(result)

    with open(output_path, "w") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    print(f"All results saved to: {output_path}")
    return results

output_file = "/content/local-qwen-output.jsonl"

results = run_model_on_dataset(
    model=model,
    tokenizer=tokenizer,
    tasks=tasks,
    output_path=output_file
)

100%|██████████| 200/200 [2:04:56<00:00, 37.48s/it]

All results saved to: /content/local-qwen-output.jsonl


#Evaluation

In [20]:
BASE_FILE = '/content/gemini-flash-lite-base-output.jsonl'
RAG_FILE = '/content/gemini-flash-lite-rag-output.jsonl'
QWEN3_14B_FILE = '/content/qwen3_14b_output.jsonl'
LOCAL_QWEN_FILE = '/content/local-qwen-output.jsonl'

In [23]:
import os
from typing import Set

def calculate_score(predicted: Set[str], golden: Set[str]) -> float:
    """AER Scoring: 1.0 (Full), 0.5 (Partial), 0.0 (Wrong)."""
    if not predicted: return 0.0
    if predicted == golden: return 1.0
    if not predicted.isdisjoint(golden): return 0.5
    return 0.0

def extract_answer_options(text: str) -> Set[str]:
    if not isinstance(text, str): return set()

    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
    content = match.group(1) if match else text

    return set(re.findall(r'\b([A-D])\b', content.upper()))

def calculate_average_score(file_path: str, response_key: str) -> float:
    total_score = 0.0
    task_count = 0

    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return 0.0

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            task = json.loads(line)

            golden_str = task.get("golden_answer", "")
            golden_set = {x.strip().upper() for x in golden_str.split(',') if x.strip()}

            raw_response = task.get(response_key, "")

            predicted_set = extract_answer_options(raw_response)
            score = calculate_score(predicted_set, golden_set)

            total_score += score
            task_count += 1

    return total_score / task_count if task_count > 0 else 0.0

In [32]:
base_score = calculate_average_score(BASE_FILE, "base_response")
print(f"1. Baseline Score: {base_score:.4f}")

1. Baseline Score: 0.6475


In [33]:
rag_score = calculate_average_score(RAG_FILE, "rag_response")
print(f"2. RAG Score:      {rag_score:.4f}")

2. RAG Score:      0.6825


In [53]:
qwen3_14b_score = calculate_average_score(QWEN3_14B_FILE, "response")
print(f"3. QWEN3-14B Score:      {qwen3_14b_score:.4f}")

3. QWEN3-14B Score:      0.6875


In [25]:
local_qwen_score = calculate_average_score(LOCAL_QWEN_FILE, "model_answer")
print(f"4. Local-QWEN3-0.6B Score:      {local_qwen_score:.4f}")

4. Local-QWEN3-0.6B Score:      0.5725


#Comparison

In [38]:
import json
import re
import os
from typing import Set, Dict, List, Tuple

BASELINE_FILE = '/content/gemini-flash-lite-base-output.jsonl'
RAG_FILE = '/content/gemini-flash-lite-rag-output.jsonl'
BASELINE_KEY = 'base_response'
RAG_KEY = 'rag_response'

IMPROVED_FILE = 'improved_list.jsonl'
DEGRADED_FILE = 'degraded_list.jsonl'

# --- Scoring Logic ---
def calculate_score(predicted: Set[str], golden: Set[str]) -> float:
    """AER Scoring: 1.0 (Full), 0.5 (Partial), 0.0 (Wrong)."""
    if not predicted: return 0.0
    if predicted == golden: return 1.0
    if not predicted.isdisjoint(golden): return 0.5
    return 0.0

# --- Data Loading---

def load_full_tasks(filepath: str) -> Dict[str, Dict]:
    results = {}

    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return results

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            task = json.loads(line)
            results[task['uuid']] = task

    return results

# --- Comparison and Reporting---

def compare_performance(baseline_tasks: Dict, rag_tasks: Dict):

    all_uuids = set(baseline_tasks.keys()) & set(rag_tasks.keys())

    baseline_total_score = 0
    rag_total_score = 0
    tasks_evaluated = 0

    rag_improvement_count = 0
    rag_degradation_count = 0

    with open(IMPROVED_FILE, 'w', encoding='utf-8') as improved_f, \
         open(DEGRADED_FILE, 'w', encoding='utf-8') as degraded_f:

        print(f"Writing improved tasks to {IMPROVED_FILE}")
        print(f"Writing degraded tasks to {DEGRADED_FILE}")

        for uuid in all_uuids:
            base_task = baseline_tasks[uuid]
            rag_task = rag_tasks[uuid]

            # 1. golden answer
            golden_str = base_task.get("golden_answer", "")
            golden_set = {x.strip().upper() for x in golden_str.split(',') if x.strip()}

            # 2. calculate baseline score
            base_raw_response = base_task.get(BASELINE_KEY, "")
            baseline_pred = extract_answer(base_raw_response)
            baseline_score = calculate_score(baseline_pred, golden_set)

            # 3. calculate RAG score
            rag_raw_response = rag_task.get(RAG_KEY, "")
            rag_pred = extract_answer(rag_raw_response)
            rag_score = calculate_score(rag_pred, golden_set)

            baseline_total_score += baseline_score
            rag_total_score += rag_score
            tasks_evaluated += 1

            if rag_score > baseline_score:
                rag_improvement_count += 1
                output_task = {
                    "uuid": uuid,
                    "response_base": base_raw_response,
                    "response_rag": rag_raw_response
                }
                improved_f.write(json.dumps(output_task, ensure_ascii=False) + '\n')

            elif rag_score < baseline_score:
                rag_degradation_count += 1
                output_task = {
                    "uuid": uuid,
                    "response_base": base_raw_response,
                    "response_rag": rag_raw_response
                }
                degraded_f.write(json.dumps(output_task, ensure_ascii=False) + '\n')

    if tasks_evaluated == 0:
        print("Error: No common tasks found for comparison.")
        return


    baseline_avg_score = baseline_total_score / tasks_evaluated
    rag_avg_score = rag_total_score / tasks_evaluated

    print("\n" + "="*50)
    print("      AER BASELINE vs RAG PERFORMANCE REPORT      ")
    print("="*50)
    print(f"Total Tasks Evaluated (Common Set): {tasks_evaluated}")
    print("-" * 50)

    print(f"1. Baseline Score (No RAG) : {baseline_avg_score:.4f}")
    print(f"2. RAG Score (with Context) : {rag_avg_score:.4f}")

    print("-" * 50)

    diff = rag_avg_score - baseline_avg_score
    diff_sign = "+" if diff >= 0 else ""
    print(f"Performance Change: {diff_sign}{diff:.4f} points.")

    print("-" * 50)
    print(f"RAG Improved Score vs. Baseline: {rag_improvement_count} tasks")
    print(f"RAG Degraded Score vs. Baseline: {rag_degradation_count} tasks")
    print("="*50)

In [39]:
baseline_tasks = load_full_tasks(BASELINE_FILE)
rag_tasks = load_full_tasks(RAG_FILE)

compare_performance(baseline_tasks, rag_tasks)

Writing improved tasks to improved_list.jsonl
Writing degraded tasks to degraded_list.jsonl

      AER BASELINE vs RAG PERFORMANCE REPORT      
Total Tasks Evaluated (Common Set): 200
--------------------------------------------------
1. Baseline Score (No RAG) : 0.6475
2. RAG Score (with Context) : 0.6825
--------------------------------------------------
Performance Change: +0.0350 points.
--------------------------------------------------
RAG Improved Score vs. Baseline: 45 tasks
RAG Degraded Score vs. Baseline: 31 tasks
